In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.nn.functional import softmax
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)
from datasets import Dataset
import yfinance as yf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

FINANCE_FILE = "Major_Data.csv"
EMOTION_FILE = "goemotions_2.csv"


In [ ]:
# =============================================================================
# STEP 1 — DIAGNOSE THE ORIGINAL BIAS
# =============================================================================
def diagnose_bias(finance_path):
    """Run this first to see how biased the original labeling was."""
    df = pd.read_csv(finance_path)
    expense_cols = ['Rent','Loan_Repayment','Insurance','Groceries','Transport',
                    'Eating_Out','Entertainment','Utilities','Healthcare','Education','Miscellaneous']
    avail = [c for c in expense_cols if c in df.columns]
    df['_total'] = df[avail].sum(axis=1)
    df['_sr']    = (df['Income'] - df['_total']) / df['Income']

    def old_label(row):
        sr = row['_sr']
        dti = row.get('Loan_Repayment', 0) / row['Income'] if row['Income'] > 0 else 0
        if sr < 0: return 'stressed'
        elif sr < 0.1 and dti > 0.3: return 'stressed'
        elif sr < 0.25: return 'neutral'
        else: return 'confident'

    labels = df.apply(old_label, axis=1)
    print("⚠️  ORIGINAL BIASED DISTRIBUTION:")
    pct = labels.value_counts(normalize=True) * 100
    for k, v in pct.items():
        bar = '█' * int(v / 2)
        print(f"   {k:12} {bar} {v:.1f}%")
    print("\n→ Model trained on this will mostly predict 'neutral' for everything!\n")
    return labels.value_counts()



In [ ]:
# =============================================================================
# STEP 2 — BALANCED DATASET INTEGRATOR
# =============================================================================
class BalancedDatasetIntegrator:
    """
    Builds a balanced 3-class dataset from:
      A) Indian Personal Finance CSV  → quantile-scored labeling
      B) GoEmotions CSV               → emotion-to-finance-class mapping
    Then undersamples to equal counts.
    """

    EXPENSE_COLS = ['Rent','Loan_Repayment','Insurance','Groceries','Transport',
                    'Eating_Out','Entertainment','Utilities','Healthcare','Education','Miscellaneous']

    # Maps 28 GoEmotions → 3 financial sentiment classes
    EMOTION_TO_FINANCE_CLASS = {
        'stressed':  ['anger','annoyance','disappointment','disapproval','disgust',
                      'embarrassment','fear','grief','nervousness','remorse','sadness'],
        'neutral':   ['neutral','confusion','realization','surprise','caring'],
        'confident': ['admiration','amusement','approval','curiosity','desire',
                      'excitement','gratitude','joy','love','optimism','pride','relief']
    }

    # Multiple text templates per class → prevents overfitting to one pattern
    TEXT_TEMPLATES = {
        'stressed': [
            "Monthly income of ₹{inc:.0f} with expenses of ₹{exp:.0f} leaving only ₹{sav:.0f} savings ({sr:.1f}%) — finances feel very tight.",
            "Earning ₹{inc:.0f}/month but after ₹{exp:.0f} in expenses only ₹{sav:.0f} remains; loan EMI of ₹{emi:.0f} is a burden.",
            "With ₹{inc:.0f} income and ₹{exp:.0f} expenses the savings rate is just {sr:.1f}% — struggling to cover basics.",
            "Financial stress: ₹{inc:.0f} income, ₹{emi:.0f} loan repayment, net savings ₹{sav:.0f} ({sr:.1f}%).",
            "Income ₹{inc:.0f}, total outgo ₹{exp:.0f}, emergency fund covers only {emg_m:.1f} months — very tight.",
            "Barely saving ₹{sav:.0f} from ₹{inc:.0f} income after all expenses; debt-to-income is unsustainable.",
        ],
        'neutral': [
            "Monthly income ₹{inc:.0f} with ₹{sav:.0f} savings ({sr:.1f}% rate) — managing adequately.",
            "Earning ₹{inc:.0f}/month, spending ₹{exp:.0f}, saving ₹{sav:.0f}; situation is stable but not growing fast.",
            "Income ₹{inc:.0f}, expenses ₹{exp:.0f}, savings rate {sr:.1f}% — okay but room for improvement.",
            "₹{inc:.0f} income, ₹{sav:.0f} monthly savings, emergency fund covers {emg_m:.1f} months — moderate position.",
            "Monthly take-home ₹{inc:.0f} after ₹{exp:.0f} expenses, {sr:.1f}% savings; loan EMI ₹{emi:.0f} is manageable.",
            "Stable finances: ₹{inc:.0f} income, {sr:.1f}% savings rate — comfortable but not building significant wealth.",
        ],
        'confident': [
            "Monthly income ₹{inc:.0f} with strong savings of ₹{sav:.0f} ({sr:.1f}%) — excellent financial health.",
            "Earning ₹{inc:.0f}/month, keeping ₹{sav:.0f} ({sr:.1f}%) after all expenses — well-positioned to invest.",
            "Income ₹{inc:.0f}, low expenses ₹{exp:.0f}, high savings ₹{sav:.0f} — {sr:.1f}% savings rate, ready to grow wealth.",
            "₹{inc:.0f} income with {emg_m:.1f} months emergency coverage and {sr:.1f}% savings — financially confident.",
            "Strong position: ₹{inc:.0f} income, ₹{sav:.0f}/month savings, minimal debt — prime time to invest.",
            "Healthy finances: {sr:.1f}% savings rate, ₹{inc:.0f} income, solid emergency fund — ready for wealth building.",
        ]
    }

    def __init__(self, finance_path, emotion_path, target_per_class=None):
        print(f"📂 Loading {finance_path}...")
        self.finance_df = pd.read_csv(finance_path)
        print(f"   → {len(self.finance_df)} financial records")

        print(f"📂 Loading {emotion_path}...")
        self.emotion_df = pd.read_csv(emotion_path)
        print(f"   → {len(self.emotion_df)} emotion samples")
        print(f"   → Columns: {list(self.emotion_df.columns[:8])} ...")
        self.target_per_class = target_per_class


In [ ]:
    # ------------------------------------------------------------------
    # GOEMOTIONS SIDE: map emotion columns → financial class
    # ------------------------------------------------------------------
    def _build_goemotions_samples(self):
        edf  = self.emotion_df.copy()
        cols = edf.columns.tolist()

        # Detect text column
        text_col = None
        for candidate in ['text', 'sentence', 'comment_text', 'comment', 'body']:
            if candidate in cols:
                text_col = candidate
                break
        if text_col is None:
            for c in cols:
                if edf[c].dtype == object:
                    text_col = c
                    break
        print(f"   GoEmotions text column: '{text_col}'")

        # Find available emotion columns
        all_emotion_names = [e for cls in self.EMOTION_TO_FINANCE_CLASS.values() for e in cls]
        avail = [c for c in all_emotion_names if c in cols]
        print(f"   Emotion columns matched: {len(avail)}/{len(all_emotion_names)}")

        if not avail:
            print("   ⚠️  No GoEmotions emotion columns found — using finance data only")
            return pd.DataFrame(columns=['text', 'label'])

        rows = []
        for fin_class, emo_list in self.EMOTION_TO_FINANCE_CLASS.items():
            matched = [c for c in emo_list if c in cols]
            if not matched:
                continue
            mask   = edf[matched].sum(axis=1) > 0
            subset = edf.loc[mask, text_col].dropna().tolist()
            for txt in subset:
                rows.append({'text': str(txt).strip(), 'label': fin_class})

        result = pd.DataFrame(rows)
        print(f"   GoEmotions label counts: {dict(Counter(result['label']))}")
        return result

In [ ]:
    # ------------------------------------------------------------------
    # COMBINE & BALANCE
    # ------------------------------------------------------------------
    def create_balanced_dataset(self, seed=42):
        print("\n🔄 Building finance samples...")
        fin_samples = self._build_finance_samples()

        print("\n🔄 Building GoEmotions samples...")
        emo_samples = self._build_goemotions_samples()

        combined = pd.concat([fin_samples, emo_samples], ignore_index=True).dropna()
        combined['text']  = combined['text'].astype(str)
        combined['label'] = combined['label'].astype(str)
        # Remove empty texts
        combined = combined[combined['text'].str.len() > 10]

        print(f"\n📊 Combined (before balance): {dict(Counter(combined['label']))}")

        # Undersample to equal class sizes
        class_counts = combined['label'].value_counts()
        min_count    = int(class_counts.min())
        n_per_class  = self.target_per_class if self.target_per_class else min_count
        n_per_class  = min(n_per_class, min_count)

        balanced_parts = []
        for cls in ['stressed', 'neutral', 'confident']:
            subset  = combined[combined['label'] == cls]
            replace = len(subset) < n_per_class   # only replace if truly needed
            sampled = subset.sample(n=n_per_class, random_state=seed, replace=replace)
            balanced_parts.append(sampled)

        balanced_df = (pd.concat(balanced_parts, ignore_index=True)
                         .sample(frac=1, random_state=seed)
                         .reset_index(drop=True))

        print(f"\n✅ BALANCED DATASET: {dict(Counter(balanced_df['label']))}")
        print(f"   Total: {len(balanced_df)} | Per class: {n_per_class}")
        total = len(balanced_df)
        for cls in ['stressed', 'neutral', 'confident']:
            c   = (balanced_df['label'] == cls).sum()
            pct = c / total * 100
            bar = '█' * int(pct / 2)
            print(f"   {cls:12} {bar} {pct:.1f}%")

        return balanced_df